In [24]:
# ==============================
# 1. Import Libraries
# ==============================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from collections import Counter

# ==============================
# 2. Load Dataset
# ==============================
df = pd.read_csv("processed_data.csv")   # change file name

print("Dataset Shape:", df.shape)
#print(df.head())

# ==============================
# 3. Handle Missing Values
# ==============================
df = df.dropna()

selected_cols = [
    'age_band_of_driver',
    'driving_experience',
    'type_of_vehicle',
    'area_accident_occured',
    'lanes_or_medians',
    'road_allignment',
    'types_of_junction',
    'road_surface_conditions',
    'light_conditions',
    'weather_conditions',
    'type_of_collision',
    'number_of_vehicles_involved',
    'number_of_casualties',
    'vehicle_movement',
    'casualty_class',
    'age_band_of_casualty',
    'pedestrian_movement',
    'cause_of_accident'
]

df = df[selected_cols + ['accident_severity']]
print("__________Dataset Shape:", df.shape)
# ==============================
# 4. Encode Categorical Data
# ==============================
global severity_encoder # Make severity_encoder globally accessible
severity_encoder = LabelEncoder()
df['accident_severity'] = severity_encoder.fit_transform(df['accident_severity'])

# Encoder for other columns (if any still exist as object type after target encoding)
le_other_cols = LabelEncoder()

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = le_other_cols.fit_transform(df[col])

# ==============================
# 5. Split Features & Target
# ==============================
X = df.drop("accident_severity", axis=1)   # change target column if needed
y = df["accident_severity"]

# ==============================
# 6. Train-Test Split
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Before SMOTE:", Counter(y_train))

# ==============================
# 7. Apply SMOTE (Augmentation)
# ==============================
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("After SMOTE:", Counter(y_train_smote))

# ==============================
# 8. Save Augmented Data (Optional)
# ==============================
augmented_data = pd.concat(
    [pd.DataFrame(X_train_smote), pd.DataFrame(y_train_smote)], axis=1
)

augmented_data.to_csv("augmented_dataset.csv", index=False)

print("Augmented dataset saved!")

Dataset Shape: (15205, 32)
__________Dataset Shape: (15205, 19)
Before SMOTE: Counter({2: 10352, 1: 1665, 0: 147})
After SMOTE: Counter({2: 10352, 1: 10352, 0: 10352})
Augmented dataset saved!


In [25]:


# ================================
# 2. LOAD DATA
# ================================
df = pd.read_csv("augmented_dataset.csv")

# ================================
# 3. CLEAN DATA
# ================================

# Remove duplicates
df = df.drop_duplicates()

# Fix NaN values
df = df.dropna()   # or use fillna if needed

# Fix inconsistent text (before encoding)
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].str.lower().str.strip()

# ================================
# 4. LABEL ENCODING
# ================================
encoders = {}

for col in df.columns:
    if df[col].dtype == 'object':
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        encoders[col] = le

print("Encoding Done")

# ================================
# 5. SPLIT X and y
# ================================
X = df.drop('accident_severity', axis=1)
y = df['accident_severity']

# ================================
# 6. APPLY SMOTE (DATA AUGMENTATION)
# ================================
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

# Convert back to DataFrame
X_res = pd.DataFrame(X_res, columns=X.columns)
y_res = pd.Series(y_res)

# ================================
# 7. FIX SMOTE FLOAT ISSUE
# ================================
int_cols = ['number_of_vehicles_involved', 'number_of_casualties']

for col in int_cols:
    if col in X_res.columns:
        X_res[col] = X_res[col].round().astype(int)

print("SMOTE Applied & Fixed")

# ================================
# 8. FINAL DATASET
# ================================
final_df = pd.concat([X_res, y_res], axis=1)

print("Final Dataset Ready")
print(final_df.head())

# ================================
# 9. SAVE FILE (OPTIONAL)
# ================================
final_df.to_csv("clean_augmented_data.csv", index=False)

Encoding Done
SMOTE Applied & Fixed
Final Dataset Ready
   age_band_of_driver  driving_experience  type_of_vehicle  \
0                   2                   0                3   
1                   0                   2                0   
2                   4                   0                7   
3                   0                   2                0   
4                   0                   0                0   

   area_accident_occured  lanes_or_medians  road_allignment  \
0                      2                 7                5   
1                     10                 3                5   
2                      7                 5                5   
3                     10                 5                5   
4                      8                 3                5   

   types_of_junction  road_surface_conditions  light_conditions  \
0                  0                        0                 3   
1                  1                        0             

In [26]:
# ================================
# 10. TRAIN MODEL
# ================================
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Split data
X = final_df.drop('accident_severity', axis=1)
y = final_df['accident_severity']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Create model
model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train model
model.fit(X_train, y_train)

print("Model Training Done")

# ================================
# 11. PREDICTION
# ================================
y_pred = model.predict(X_test)

# ================================
# 12. EVALUATION
# ================================
print("\nAccuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

print("\nClassification Report:\n", classification_report(y_test, y_pred))

Model Training Done

Accuracy: 0.9231457657456091

Confusion Matrix:
 [[2977    5   15]
 [  13 2707  286]
 [  25  343 2568]]

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99      2997
           1       0.89      0.90      0.89      3006
           2       0.90      0.87      0.88      2936

    accuracy                           0.92      8939
   macro avg       0.92      0.92      0.92      8939
weighted avg       0.92      0.92      0.92      8939



In [27]:
new_data = X_test.iloc[2:3]
print(new_data)

pred = model.predict(new_data)
print("Prediction:", pred)

       age_band_of_driver  driving_experience  type_of_vehicle  \
14321                   1                   2                5   

       area_accident_occured  lanes_or_medians  road_allignment  \
14321                      7                 3                5   

       types_of_junction  road_surface_conditions  light_conditions  \
14321                  1                        0                 2   

       weather_conditions  type_of_collision  number_of_vehicles_involved  \
14321                   2                  8                            2   

       number_of_casualties  vehicle_movement  casualty_class  \
14321                     1                 2               0   

       age_band_of_casualty  pedestrian_movement  cause_of_accident  
14321                     1                    5                  9  
Prediction: [0]


In [28]:
new_data = X_test.iloc[29:30]
print(new_data)

pred = model.predict(new_data)
print("Prediction:", pred)

      age_band_of_driver  driving_experience  type_of_vehicle  \
7976                   0                   4                6   

      area_accident_occured  lanes_or_medians  road_allignment  \
7976                      8                 3                5   

      types_of_junction  road_surface_conditions  light_conditions  \
7976                  7                        3                 0   

      weather_conditions  type_of_collision  number_of_vehicles_involved  \
7976                   4                  8                            1   

      number_of_casualties  vehicle_movement  casualty_class  \
7976                     4                 2               0   

      age_band_of_casualty  pedestrian_movement  cause_of_accident  
7976                     1                    5                 12  
Prediction: [2]


In [29]:
new_data = X_test.iloc[1100:1101]
print(new_data)

pred = model.predict(new_data)
print("Prediction:", pred)

       age_band_of_driver  driving_experience  type_of_vehicle  \
25396                   0                   1                5   

       area_accident_occured  lanes_or_medians  road_allignment  \
25396                      8                 1                3   

       types_of_junction  road_surface_conditions  light_conditions  \
25396                  7                        0                 0   

       weather_conditions  type_of_collision  number_of_vehicles_involved  \
25396                   2                  2                            2   

       number_of_casualties  vehicle_movement  casualty_class  \
25396                     1                 4               1   

       age_band_of_casualty  pedestrian_movement  cause_of_accident  
25396                     3                    5                  0  
Prediction: [1]


In [30]:
new_data = pd.DataFrame([[
    2,  # age_band_of_driver
    1,  # driving_experience
    0,  # type_of_vehicle
    3,  # area_accident_occured
    1,  # lanes_or_medians
    2,  # road_allignment
    0,  # types_of_junction
    1,  # road_surface_conditions
    2,  # light_conditions
    0,  # weather_conditions
    1,  # type_of_collision
    3,  # number_of_vehicles_involved
    5,  # number_of_casualties
    1,  # vehicle_movement
    0,  # casualty_class
    1,  # age_band_of_casualty
    0,  # pedestrian_movement
    2   # cause_of_accident
]], columns=[
    'age_band_of_driver',
    'driving_experience',
    'type_of_vehicle',
    'area_accident_occured',
    'lanes_or_medians',
    'road_allignment',
    'types_of_junction',
    'road_surface_conditions',
    'light_conditions',
    'weather_conditions',
    'type_of_collision',
    'number_of_vehicles_involved',
    'number_of_casualties',
    'vehicle_movement',
    'casualty_class',
    'age_band_of_casualty',
    'pedestrian_movement',
    'cause_of_accident'
])

In [31]:
new_datas = pd.DataFrame([[
    3,  # age_band_of_driver
    1,  # driving_experience (low)
    0,  # type_of_vehicle
    2,  # area_accident_occured
    1,  # lanes_or_medians
    2,  # road_allignment
    0,  # types_of_junction
    1,  # road_surface_conditions
    2,  # light_conditions (dark)
    0,  # weather_conditions (normal)
    1,  # type_of_collision
    3,  # number_of_vehicles_involved
    4,  # number_of_casualties (high)
    1,  # vehicle_movement
    0,  # casualty_class
    2,  # age_band_of_casualty
    0,  # pedestrian_movement
    2   # cause_of_accident (careless)
]], columns=[
    'age_band_of_driver',
    'driving_experience',
    'type_of_vehicle',
    'area_accident_occured',
    'lanes_or_medians',
    'road_allignment',
    'types_of_junction',
    'road_surface_conditions',
    'light_conditions',
    'weather_conditions',
    'type_of_collision',
    'number_of_vehicles_involved',
    'number_of_casualties',
    'vehicle_movement',
    'casualty_class',
    'age_band_of_casualty',
    'pedestrian_movement',
    'cause_of_accident'
])

In [32]:
new_data2 = pd.DataFrame([[
    1,  # age_band_of_driver (young)
    3,  # driving_experience (good)
    0,  # type_of_vehicle
    3,  # area_accident_occured (safe)
    1,  # lanes_or_medians
    2,  # road_allignment
    0,  # types_of_junction
    1,  # road_surface_conditions (dry)
    3,  # light_conditions (daylight)
    2,  # weather_conditions (normal)
    0,  # type_of_collision (minor)
    1,  # number_of_vehicles_involved
    1,  # number_of_casualties (low)
    0,  # vehicle_movement
    0,  # casualty_class
    1,  # age_band_of_casualty
    0,  # pedestrian_movement
    1   # cause_of_accident (normal)
]], columns=[
    'age_band_of_driver',
    'driving_experience',
    'type_of_vehicle',
    'area_accident_occured',
    'lanes_or_medians',
    'road_allignment',
    'types_of_junction',
    'road_surface_conditions',
    'light_conditions',
    'weather_conditions',
    'type_of_collision',
    'number_of_vehicles_involved',
    'number_of_casualties',
    'vehicle_movement',
    'casualty_class',
    'age_band_of_casualty',
    'pedestrian_movement',
    'cause_of_accident'
])

In [33]:
prediction = model.predict(new_data)

label = severity_encoder.inverse_transform(prediction)

print("Prediction:", label[0])

Prediction: Serious Injury
